# WO-03 Q1b — bearing origin
The ring resolver measures `border_bearing` and `centroid_bearing` from the core basin's centroid, not from the place's own point. This notebook recomputes the bearings from the place point and compares, in degrees and in compass words. Needs `output/edop/whg/q1_pairs.csv`, written by Cell 12 of `wo03_q1_ring.ipynb`. Read-only.

In [1]:
%matplotlib inline
# Cell 1 — Setup and load the pair table from Q1 Cell 12
import sys, math, warnings
warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy")
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts' / 'edop').exists())
sys.path.insert(0, str(ROOT))
from scripts.shared.db_utils import db_connect
from scripts.edop.areas import engine as E
from scripts.edop.whg import wo03_common as C

conn = db_connect()
PAIRS = pd.read_csv(C.OUT / 'q1_pairs.csv', dtype={'tag': str})
print('commit', C.commit(), '| pairs loaded:', len(PAIRS), '|', PAIRS['source'].value_counts().to_dict())
print('columns:', list(PAIRS.columns))

commit d0bb073 | pairs loaded: 1739 | {'sample': 1680, 'test': 59}
columns: ['source', 'tag', 'core_id', 'nb_id', 'nb_km2', 'shared_km', 'border_bearing', 'centroid_bearing', 'mlat', 'mlon', 'inter_type', 'dim', 'border_m', 'core_perimeter_m', 'mid_to_border_m', 'bearing_gap', 'border_share_of_perimeter']


In [2]:
# Cell 2 — Bearings from the place point, versus the engine's bearings from the core centroid
# Sample points are ST_PointOnSurface of the core (WO-02 harvest): inside the basin but not at the centroid, a stand-in for a settlement.
sample_pt = {int(p['hybas_id_l6']): (p['lat'], p['lon']) for p in C.population_points()}
def place_point(row):
    return C.PLACES[row['tag']] if row['source'] == 'test' else sample_pt[int(row['tag'])]
pp = PAIRS.apply(place_point, axis=1)
PAIRS['plat'] = [p[0] for p in pp]; PAIRS['plon'] = [p[1] for p in pp]

# Closest point on the shared border to the place, and the place's distance to the border (geography; PostGIS 3.6)
sql = """
SELECT p.core_id, p.nb_id,
       ST_Y(cp.g) AS cp_lat, ST_X(cp.g) AS cp_lon,
       ST_Distance(i.pt::geography, i.g::geography) AS place_to_border_m
FROM unnest(%s::bigint[], %s::bigint[], %s::float8[], %s::float8[]) AS p(core_id, nb_id, plon, plat)
JOIN public.basin06 a ON a.hybas_id = p.core_id
JOIN public.basin06 b ON b.hybas_id = p.nb_id
CROSS JOIN LATERAL (SELECT ST_Intersection(a.geom, b.geom) AS g, ST_SetSRID(ST_MakePoint(p.plon, p.plat), 4326) AS pt) i
CROSS JOIN LATERAL (SELECT ST_ClosestPoint(i.g::geography, i.pt::geography)::geometry AS g) cp
"""
key = PAIRS[['core_id', 'nb_id', 'plon', 'plat']].drop_duplicates(['core_id', 'nb_id'])
fx = pd.DataFrame(conn.execute(sql, (key['core_id'].tolist(), key['nb_id'].tolist(), key['plon'].tolist(), key['plat'].tolist())).fetchall(),
                  columns=['core_id', 'nb_id', 'cp_lat', 'cp_lon', 'place_to_border_m'])
fx['core_id'] = fx['core_id'].astype('int64'); fx['nb_id'] = fx['nb_id'].astype('int64')
B = PAIRS.merge(fx, on=['core_id', 'nb_id'], how='left')

# Core centroid (the engine's bearing origin) and whether it lies inside the core
cc = pd.DataFrame(conn.execute(
    "SELECT hybas_id::bigint, ST_Y(ST_Centroid(geom)), ST_X(ST_Centroid(geom)), ST_Contains(geom, ST_Centroid(geom)), sub_area "
    "FROM public.basin06 WHERE hybas_id = ANY(%s)", (B['core_id'].unique().tolist(),)).fetchall(),
    columns=['core_id', 'clat', 'clon', 'centroid_inside', 'core_km2'])
cc['core_id'] = cc['core_id'].astype('int64')
B = B.merge(cc, on='core_id', how='left')

def bear(r, la2, lo2):
    return E._bearing(r['plat'], r['plon'], r[la2], r[lo2])
B['b_engine'] = B['border_bearing']                                              # core centroid -> border_mid
B['b_place_mid'] = B.apply(lambda r: bear(r, 'mlat', 'mlon'), axis=1)            # place -> border_mid
B['b_place_closest'] = B.apply(lambda r: bear(r, 'cp_lat', 'cp_lon'), axis=1)    # place -> closest point on the shared border
B['b_centroid_engine'] = B['centroid_bearing']                                   # core centroid -> neighbour interior point
def cdiff(a, b):
    d = (a - b).abs() % 360
    return np.minimum(d, 360 - d)
B['d_engine_vs_place_mid'] = cdiff(B['b_engine'], B['b_place_mid'])
B['d_engine_vs_place_closest'] = cdiff(B['b_engine'], B['b_place_closest'])
B['d_centroidb_vs_place_closest'] = cdiff(B['b_centroid_engine'], B['b_place_closest'])
N8 = ['N', 'NE', 'E', 'SE', 'S', 'SW', 'W', 'NW']; N4 = ['N', 'E', 'S', 'W']
def s8(b): return N8[int(((b + 22.5) % 360) // 45)]
def s4(b): return N4[int(((b + 45) % 360) // 90)]
for c in ['b_engine', 'b_place_mid', 'b_place_closest', 'b_centroid_engine']:
    B['s8_' + c] = B[c].apply(s8); B['s4_' + c] = B[c].apply(s4)
B.to_csv(C.OUT / 'q1_bearing_origin.csv', index=False)
print('pairs', len(B), '| with no closest-point row:', int(B['cp_lat'].isna().sum()), '| saved q1_bearing_origin.csv')

pairs 1739 | with no closest-point row: 0 | saved q1_bearing_origin.csv


In [3]:
# Cell 3 — How far the origin moves, and how often the bearing and the compass word change
def hav_km(la1, lo1, la2, lo2):
    p1, p2 = math.radians(la1), math.radians(la2); dl = math.radians(lo2 - lo1)
    a = math.sin((p2 - p1) / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * 6371.0088 * math.asin(math.sqrt(a))
cu = B.drop_duplicates('core_id').copy()
cu['origin_shift_km'] = [hav_km(r.plat, r.plon, r.clat, r.clon) for r in cu.itertuples()]
cu['shift_over_core_width'] = cu['origin_shift_km'] / np.sqrt(cu['core_km2'])

L = []
L.append(f'cores: {len(cu)} | core centroid lies outside its own core: {int((~cu["centroid_inside"].astype(bool)).sum())}')
L.append('distance from the place point to the core centroid (km), and as a share of sqrt(core area):')
L.append(cu[['origin_shift_km', 'shift_over_core_width']].describe(percentiles=[.1, .5, .9]).round(3).to_string())
L.append('')
L.append('circular difference in degrees between bearings:')
L.append(B[['d_engine_vs_place_mid', 'd_engine_vs_place_closest', 'd_centroidb_vs_place_closest']].describe(percentiles=[.5, .9, .99]).round(1).to_string())
L.append('')
rows = []
for name, a, b in [('engine border bearing vs place->border_mid', 'b_engine', 'b_place_mid'),
                   ('engine border bearing vs place->closest border point', 'b_engine', 'b_place_closest'),
                   ('engine centroid bearing vs place->closest border point', 'b_centroid_engine', 'b_place_closest'),
                   ('engine border bearing vs engine centroid bearing', 'b_engine', 'b_centroid_engine')]:
    for grp, sub in [('all', B), ('test', B[B.source == 'test']), ('sample', B[B.source == 'sample']),
                     ('centroid outside core', B[~B['centroid_inside'].astype(bool)])]:
        rows.append(dict(comparison=name, pairs=grp, n=len(sub),
                         pct_8pt_differs=round(100 * (sub['s8_' + a] != sub['s8_' + b]).mean(), 1),
                         pct_4pt_differs=round(100 * (sub['s4_' + a] != sub['s4_' + b]).mean(), 1)))
L.append('share of pairs whose compass word differs (8-point and 4-point):')
L.append(pd.DataFrame(rows).to_string(index=False))
L.append('')
L.append('distance from the place point to the shared border (km):')
L.append((B['place_to_border_m'] / 1000).describe(percentiles=[.1, .5, .9]).round(2).to_string())
print('\n'.join(L))

cores: 307 | core centroid lies outside its own core: 10
distance from the place point to the core centroid (km), and as a share of sqrt(core area):
       origin_shift_km  shift_over_core_width
count          307.000                307.000
mean            14.695                  0.190
std             18.981                  0.231
min              0.209                  0.004
10%              2.102                  0.048
50%              8.349                  0.112
90%             33.126                  0.375
max            127.482                  1.615

circular difference in degrees between bearings:
       d_engine_vs_place_mid  d_engine_vs_place_closest  d_centroidb_vs_place_closest
count                 1739.0                     1739.0                        1739.0
mean                    10.5                       21.1                          25.7
std                     17.7                       22.0                          25.8
min                      0.0               

In [4]:
# Cell 4 — The nine test places: 8-point compass word per neighbour under each method
t = B[B.source == 'test'].sort_values(['tag', 'nb_km2'], ascending=[True, False])
out = t[['tag', 'nb_id', 'shared_km', 's8_b_engine', 's8_b_place_mid', 's8_b_place_closest', 's8_b_centroid_engine']].rename(columns={
    's8_b_engine': 'engine_border', 's8_b_place_mid': 'place_to_mid', 's8_b_place_closest': 'place_to_closest', 's8_b_centroid_engine': 'engine_centroid'})
print('engine_border = from core centroid to border midpoint (engine); place_to_mid = from the place to the same midpoint;')
print('place_to_closest = from the place to the nearest point of the shared border; engine_centroid = core centroid to neighbour interior point (engine)')
print()
print(out.to_string(index=False))

engine_border = from core centroid to border midpoint (engine); place_to_mid = from the place to the same midpoint;
place_to_closest = from the place to the nearest point of the shared border; engine_centroid = core centroid to neighbour interior point (engine)

         tag      nb_id  shared_km engine_border place_to_mid place_to_closest engine_centroid
  Alexandria 1060034260     336.09             E           SE                E              SE
  Alexandria 1060034270       3.14             N           NE               NE               E
  Alexandria 1060044570     234.69             S            S                S              SW
  Alexandria 1060043560      97.72             S            S               SE               S
  Alexandria 1060034040     102.03             W           SW               SW               W
Chaco Canyon 7060674060     118.61            SE           SE               SE              SE
Chaco Canyon 7060607440      64.18            NW           NW           